In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Finetunning on HP

In [3]:
!git clone https://github.com/abhijitdalal26/harry-potter-gpt.git

Cloning into 'harry-potter-gpt'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 74 (delta 30), reused 66 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 5.59 MiB | 17.66 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [4]:
%cd /kaggle/working/harry-potter-gpt/nanoGPT

/kaggle/working/harry-potter-gpt/nanoGPT


In [5]:
!pwd

/kaggle/working/harry-potter-gpt/nanoGPT


In [ ]:
!python sample.py --init_from=gpt2 --start="Harry Potter" --num_samples=3 --max_new_tokens=200

In [ ]:
!torchrun --standalone --nproc_per_node=2 train.py config/finetune_harry_potter.py

In [ ]:
!python sample.py --out_dir=out-harry-potter --start="Harry Potter" --num_samples=3 --max_new_tokens=200

In [ ]:
# zip the folder
!zip -r /kaggle/working/out-harry-potter.zip /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter

# download the zip file
from IPython.display import FileLink
FileLink('/kaggle/working/out-harry-potter.zip')

In [ ]:
!pip install -q PyDrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from google.colab import auth

# authenticate
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# upload file
file = drive.CreateFile({'title': 'out-harry-potter.zip'})
file.SetContentFile('/kaggle/working/out-harry-potter.zip')
file.Upload()

print("Uploaded to Google Drive")

## SFT

In [6]:
!gdown https://drive.google.com/uc?id=1VCeBlg5rz3sduwRaLtdZLH3-Ic30gDuQ

Downloading...
From (original): https://drive.google.com/uc?id=1VCeBlg5rz3sduwRaLtdZLH3-Ic30gDuQ
From (redirected): https://drive.google.com/uc?id=1VCeBlg5rz3sduwRaLtdZLH3-Ic30gDuQ&confirm=t&uuid=b768a2fa-331d-4ba9-8cd0-c1ac1774c25f
To: /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter.zip
100%|███████████████████████████████████████| 1.38G/1.38G [00:13<00:00, 103MB/s]


In [7]:
!unzip -j /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter.zip -d /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter/

Archive:  /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter.zip
  inflating: /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter/ckpt.pt  


In [8]:
!torchrun --standalone --nproc_per_node=2 train.py config/harry_potter_sft.py

W0329 06:35:59.086000 117 torch/distributed/run.py:852] 
W0329 06:35:59.086000 117 torch/distributed/run.py:852] *****************************************
W0329 06:35:59.086000 117 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0329 06:35:59.086000 117 torch/distributed/run.py:852] *****************************************
[W329 06:35:59.978499696 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Overriding config with config/harry_potter_sft.py:Overriding config with config/harry_potter_sft.py:

import time

out_dir = 'out-harry-potter'
eval_interval = 50
eval_iters = 20
wandb_log = False
wandb_project = 'harry-potter'
wandb_run_name = 'sft-' + str(time.time())

dataset = 'harry_potter_sft'     # points to data/harry_potter_sft/
init_from = 'resume'           

In [9]:
!python sample.py \
    --out_dir=/kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter \
    --start="<|user|> Who is Harry Potter?\n<|assistant|>" \
    --max_new_tokens=200 \
    --num_samples=3 \
    --temperature=0.8 \
    --top_k=200 \
    --device=cuda

Overriding: out_dir = /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter
Overriding: start = <|user|> Who is Harry Potter?\n<|assistant|>
Overriding: max_new_tokens = 200
Overriding: num_samples = 3
Overriding: temperature = 0.8
Overriding: top_k = 200
Overriding: device = cuda
number of parameters: 123.65M
No meta.pkl found, assuming GPT-2 encodings...
<|user|> Who is Harry Potter?\n<|assistant|> He’s the one who stops the Ministry so Dumbledore can walk in and stop the war. Harry’s biggest mistake was thinking he was protecting Hermione, but then he’s actually supporting the Death Eaters. Do you think Dumbledore would have won if Harry had known?
<|endoftext|>A recent piece by the Carefree Media Collective's Jordan Loy explains how the Department for International Development (DfID) has been "engaged" in the "war on youth"—directly or through the Department of Defense (DfD)—because of the war on poverty. The Department of Defense has also been covertly supporting the "scandal"

In [10]:
!python sample.py \
    --out_dir=/kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter \
    --start="<|user|> Does snape trying to save Harry Potter?\n<|assistant|>" \
    --max_new_tokens=200 \
    --num_samples=3 \
    --temperature=0.8 \
    --top_k=200 \
    --device=cuda

Overriding: out_dir = /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter
Overriding: start = <|user|> Does snape trying to save Harry Potter?\n<|assistant|>
Overriding: max_new_tokens = 200
Overriding: num_samples = 3
Overriding: temperature = 0.8
Overriding: top_k = 200
Overriding: device = cuda
number of parameters: 123.65M
No meta.pkl found, assuming GPT-2 encodings...
<|user|> Does snape trying to save Harry Potter?\n<|assistant|> That moment feels like such a huge reveal. I didn’t fully understand it until rereading the whole series later. Did that moment feel like more of a test for you than anything?
<|endoftext|>It was the year the Ministry of Magic started handing out Death Eaters. Law and Order of the Phoenix were the top three, and the Ministry dropped the Death Eater money until the Ministry realized they were a fraud. On re-read, it becomes clear just how much loyalty they earned on an open-ended cost.
<|endoftext|>The first person to go to the zoo in London was a c

In [11]:
!zip -r /kaggle/working/out-harry-potter-sft.zip /kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter

  adding: kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter/ (stored 0%)
  adding: kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter/ckpt.pt (deflated 7%)


In [12]:
!pip install -q PyDrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from google.colab import auth

# authenticate
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# upload file
file = drive.CreateFile({'title': 'out-harry-potter-sft.zip'})
file.SetContentFile('/kaggle/working/out-harry-potter-sft.zip')
file.Upload()

print("Uploaded to Google Drive")

Uploaded to Google Drive


## Convert nanoGPT to Hugging Face

In [15]:
"""
Converts a nanoGPT ckpt.pt to a HuggingFace GPT2 model.
Usage: python convert_to_hf.py
"""

import torch
from transformers import GPT2LMHeadModel, GPT2Config, GPT2Tokenizer

# ── config ────────────────────────────────────────────────────────────────────
CKPT_PATH  = '/kaggle/working/harry-potter-gpt/nanoGPT/out-harry-potter/ckpt.pt'
OUTPUT_DIR = '/kaggle/working/harry-potter-hf'
# ─────────────────────────────────────────────────────────────────────────────

print("Loading checkpoint...")
ckpt = torch.load(CKPT_PATH, map_location='cpu')
model_args = ckpt['model_args']
state_dict = ckpt['model']

print("Model args:", model_args)

# fix the keys — nanoGPT sometimes saves with _orig_mod. prefix
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

# map nanoGPT keys → HuggingFace GPT2 keys
key_mapping = {
    'transformer.wte.weight':       'transformer.wte.weight',
    'transformer.wpe.weight':       'transformer.wpe.weight',
    'transformer.ln_f.weight':      'transformer.ln_f.weight',
    'transformer.ln_f.bias':        'transformer.ln_f.bias',
    'lm_head.weight':               'lm_head.weight',
}

# map attention + mlp + layernorm blocks
for i in range(model_args['n_layer']):
    key_mapping.update({
        # layernorms
        f'transformer.h.{i}.ln_1.weight':           f'transformer.h.{i}.ln_1.weight',
        f'transformer.h.{i}.ln_1.bias':             f'transformer.h.{i}.ln_1.bias',
        f'transformer.h.{i}.ln_2.weight':           f'transformer.h.{i}.ln_2.weight',
        f'transformer.h.{i}.ln_2.bias':             f'transformer.h.{i}.ln_2.bias',
        # attention
        f'transformer.h.{i}.attn.c_attn.weight':    f'transformer.h.{i}.attn.c_attn.weight',
        f'transformer.h.{i}.attn.c_attn.bias':      f'transformer.h.{i}.attn.c_attn.bias',
        f'transformer.h.{i}.attn.c_proj.weight':    f'transformer.h.{i}.attn.c_proj.weight',
        f'transformer.h.{i}.attn.c_proj.bias':      f'transformer.h.{i}.attn.c_proj.bias',
        # mlp
        f'transformer.h.{i}.mlp.c_fc.weight':       f'transformer.h.{i}.mlp.c_fc.weight',
        f'transformer.h.{i}.mlp.c_fc.bias':         f'transformer.h.{i}.mlp.c_fc.bias',
        f'transformer.h.{i}.mlp.c_proj.weight':     f'transformer.h.{i}.mlp.c_proj.weight',
        f'transformer.h.{i}.mlp.c_proj.bias':       f'transformer.h.{i}.mlp.c_proj.bias',
    })

# these keys need to be transposed (nanoGPT uses Conv1D, HF uses Linear)
transposed_keys = [
    'attn.c_attn.weight',
    'attn.c_proj.weight',
    'mlp.c_fc.weight',
    'mlp.c_proj.weight',
]

# remap + transpose state dict
new_state_dict = {}
for nano_key, hf_key in key_mapping.items():
    if nano_key in state_dict:
        tensor = state_dict[nano_key]
        if any(nano_key.endswith(k) for k in transposed_keys):
            print(f"Transposing {nano_key}: {tensor.shape} → {tensor.t().shape}")
            tensor = tensor.t()
        new_state_dict[hf_key] = tensor
    else:
        print(f"WARNING: missing key {nano_key}")

# build HF config from nanoGPT model args
hf_config = GPT2Config(
    vocab_size      = model_args['vocab_size'],
    n_positions     = model_args['block_size'],
    n_embd          = model_args['n_embd'],
    n_layer         = model_args['n_layer'],
    n_head          = model_args['n_head'],
    resid_pdrop     = 0.0,
    embd_pdrop      = 0.0,
    attn_pdrop      = 0.0,
    use_cache       = True,
)

print("Building HuggingFace model...")
hf_model = GPT2LMHeadModel(hf_config)
hf_model.load_state_dict(new_state_dict, strict=False)

print(f"Saving to {OUTPUT_DIR} ...")
hf_model.save_pretrained(OUTPUT_DIR)

# save tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nDone! Model saved to:", OUTPUT_DIR)
print("\nVerify with:")
print("  from transformers import AutoModelForCausalLM, AutoTokenizer")
print(f"  model = AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR}')")

Loading checkpoint...
Model args: {'n_layer': 12, 'n_head': 12, 'n_embd': 768, 'block_size': 1024, 'bias': True, 'vocab_size': 50257, 'dropout': 0.1}
Transposing transformer.h.0.attn.c_attn.weight: torch.Size([2304, 768]) → torch.Size([768, 2304])
Transposing transformer.h.0.attn.c_proj.weight: torch.Size([768, 768]) → torch.Size([768, 768])
Transposing transformer.h.0.mlp.c_fc.weight: torch.Size([3072, 768]) → torch.Size([768, 3072])
Transposing transformer.h.0.mlp.c_proj.weight: torch.Size([768, 3072]) → torch.Size([3072, 768])
Transposing transformer.h.1.attn.c_attn.weight: torch.Size([2304, 768]) → torch.Size([768, 2304])
Transposing transformer.h.1.attn.c_proj.weight: torch.Size([768, 768]) → torch.Size([768, 768])
Transposing transformer.h.1.mlp.c_fc.weight: torch.Size([3072, 768]) → torch.Size([768, 3072])
Transposing transformer.h.1.mlp.c_proj.weight: torch.Size([768, 3072]) → torch.Size([3072, 768])
Transposing transformer.h.2.attn.c_attn.weight: torch.Size([2304, 768]) → torc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Done! Model saved to: /kaggle/working/harry-potter-hf

Verify with:
  from transformers import AutoModelForCausalLM, AutoTokenizer
  model = AutoModelForCausalLM.from_pretrained('/kaggle/working/harry-potter-hf')


In [17]:
# Cell 3: Verify the conversion

import os
print("Files in output directory:")
for file in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, file)
    size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
    print(f"  {file}: {size:.2f} MB")

Files in output directory:
  config.json: 0.00 MB
  generation_config.json: 0.00 MB
  model.safetensors: 474.71 MB
  tokenizer_config.json: 0.00 MB
  tokenizer.json: 3.39 MB


In [18]:
# Cell 4: Test the converted model

from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading converted model...")
model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
print("Model loaded successfully!")

# Show model info
print(f"\nModel Info:")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Config: {model.config}")

Loading converted model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!

Model Info:
  Parameters: 124,439,808
  Config: GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.0,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 50257
}



In [19]:
# Cell 5: Generate text with the model

from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

model.eval()

# Test generation
test_prompts = [
    "Harry Potter was",
    "The magical world",
    "Hogwarts School of"
]

print("Text Generation Test:\n")
print("="*60)

for prompt in test_prompts:
    print(f"\nPrompt: '{prompt}'")
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=50,
            num_beams=1,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Generated: '{generated_text}'")
    print("-"*60)

print("\n✓ Generation test complete!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Text Generation Test:


Prompt: 'Harry Potter was'
Generated: 'Harry Potter was a wizard, but he also had this very specific love for magic that was completely absent from his other books. I didn't realize until later that Harry was also a wizard with the ability to control the weather. Do you think that connection'
------------------------------------------------------------

Prompt: 'The magical world'
Generated: 'The magical world of Harry and Ron is one where magic is both real and dangerous. The books treat it like the mystery of life and death that only Harry can see, and Ron is the only one who knows.
'
------------------------------------------------------------

Prompt: 'Hogwarts School of'
Generated: 'Hogwarts School of Witchcraft and Wizardry). The story is full of clever logic and logic, but Rowling never shows it explicitly. What makes her shift so dramatic is that she chooses to leave it open for the audience to interpret.
'
-----------------------------------------------

In [20]:
!zip -r /kaggle/working/harry-potter-hf.zip /kaggle/working/harry-potter-hf

  adding: kaggle/working/harry-potter-hf/ (stored 0%)
  adding: kaggle/working/harry-potter-hf/config.json (deflated 52%)
  adding: kaggle/working/harry-potter-hf/generation_config.json (deflated 36%)
  adding: kaggle/working/harry-potter-hf/model.safetensors (deflated 7%)
  adding: kaggle/working/harry-potter-hf/tokenizer_config.json (deflated 48%)
  adding: kaggle/working/harry-potter-hf/tokenizer.json (deflated 82%)


In [22]:
!pip install -q PyDrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials
from google.colab import auth

# authenticate
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# upload file
file = drive.CreateFile({'title': 'harry-potter-hf.zip'})
file.SetContentFile('/kaggle/working/harry-potter-hf.zip')
file.Upload()

print("Uploaded to Google Drive")

Uploaded to Google Drive
